In [21]:
from __future__ import division, print_function, unicode_literals
from pathlib import Path
import random
import numpy as np

def accuracy_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(y_true == y_pred)

def _logsumexp(values, axis=1):
    max_values = np.max(values, axis=axis, keepdims=True)
    stable = values - max_values
    summed = np.sum(np.exp(stable), axis=axis, keepdims=True)
    return (max_values + np.log(summed)).squeeze(axis=axis)

class MultinomialNB:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)

        self.classes_, class_inverse = np.unique(y, return_inverse=True)
        n_classes = len(self.classes_)
        n_features = X.shape[1]

        class_count = np.zeros(n_classes, dtype=float)
        feature_count = np.zeros((n_classes, n_features), dtype=float)

        for class_idx in range(n_classes):
            mask = class_inverse == class_idx
            X_class = X[mask]
            class_count[class_idx] = X_class.shape[0]
            feature_count[class_idx] = X_class.sum(axis=0)

        smoothed_fc = feature_count + self.alpha
        smoothed_cc = smoothed_fc.sum(axis=1, keepdims=True)

        self.class_log_prior_ = np.log(class_count / class_count.sum())
        self.feature_log_prob_ = np.log(smoothed_fc / smoothed_cc)
        return self

    def _joint_log_likelihood(self, X):
        X = np.asarray(X, dtype=float)
        return X @ self.feature_log_prob_.T + self.class_log_prior_

    def predict_log_proba(self, X):
        jll = self._joint_log_likelihood(X)
        normalizer = _logsumexp(jll, axis=1)[:, None]
        return jll - normalizer

    def predict_proba(self, X):
        return np.exp(self.predict_log_proba(X))

    def predict(self, X):
        jll = self._joint_log_likelihood(X)
        return self.classes_[np.argmax(jll, axis=1)]

class BernoulliNB:
    def __init__(self, alpha=1.0, binarize=0.0):
        self.alpha = alpha
        self.binarize = binarize

    def _binarize(self, X):
        X = np.asarray(X, dtype=float)
        if self.binarize is None:
            return X
        return (X > self.binarize).astype(float)

    def fit(self, X, y):
        X = self._binarize(X)
        y = np.asarray(y)

        self.classes_, class_inverse = np.unique(y, return_inverse=True)
        n_classes = len(self.classes_)
        n_features = X.shape[1]

        class_count = np.zeros(n_classes, dtype=float)
        feature_count = np.zeros((n_classes, n_features), dtype=float)

        for class_idx in range(n_classes):
            mask = class_inverse == class_idx
            X_class = X[mask]
            class_count[class_idx] = X_class.shape[0]
            feature_count[class_idx] = X_class.sum(axis=0)

        smoothed_fc = feature_count + self.alpha
        smoothed_cc = class_count[:, None] + 2*self.alpha

        self.class_log_prior_ = np.log(class_count / class_count.sum())
        self.feature_prob_ = smoothed_fc / smoothed_cc
        self.feature_log_prob_ = np.log(self.feature_prob_)
        self.feature_log_prob_neg_ = np.log(1.0 - self.feature_prob_)
        return self

    def _joint_log_likelihood(self, X):
        X = self._binarize(X)
        positive = X @ self.feature_log_prob_.T
        negative = (1.0 - X) @ self.feature_log_prob_neg_.T
        return positive + negative + self.class_log_prior_

    def predict_log_proba(self, X):
        jll = self._joint_log_likelihood(X)
        normalizer = _logsumexp(jll, axis=1)[:, None]
        return jll - normalizer

    def predict_proba(self, X):
        return np.exp(self.predict_log_proba(X))

    def predict(self, X):
        jll = self._joint_log_likelihood(X)
        return self.classes_[np.argmax(jll, axis=1)]


## Ví dụ 1: Bắc hay Nam với Multinomial Naive Bayes

Đây là ví dụ nhỏ nhất để hiểu mô hình đang làm gì.

Mỗi câu được biểu diễn bằng một vector đếm số lần xuất hiện của các từ. Vì thế `MultinomialNB` hợp lý hơn, vì nó làm việc tốt với dữ liệu dạng **đếm tần suất**.

Trong code dưới đây:
- `d1, d2, d3, d4` là dữ liệu huấn luyện,
- `label` là nhãn `B` hoặc `N`,
- `d5, d6` là dữ liệu mới để mô hình dự đoán.

Ý chính là mô hình sẽ học xem lớp nào thường đi kèm với những từ nào nhiều hơn.

In [22]:
# train data
d1 = [2, 1, 1, 0, 0, 0, 0, 0, 0]
d2 = [1, 1, 0, 1, 1, 0, 0, 0, 0]
d3 = [0, 1, 0, 0, 1, 1, 0, 0, 0]
d4 = [0, 1, 0, 0, 0, 0, 1, 1, 1]
train_data = np.array([d1, d2, d3, d4])
label = np.array(['B', 'B', 'B', 'N'])

# test data
d5 = np.array([[2, 0, 0, 1, 0, 0, 0, 1, 0]])
d6 = np.array([[0, 1, 0, 0, 0, 0, 0, 1, 1]])

clf = MultinomialNB()
clf.fit(train_data, label)

print('Predicting class of d5:', str(clf.predict(d5)[0]))
print('Probability of d6 in each class:', clf.predict_proba(d6))

Predicting class of d5: B
Probability of d6 in each class: [[0.29175335 0.70824665]]


## Ví dụ 2: Bắc hay Nam với Bernoulli Naive Bayes

Ở ví dụ này, thay vì đếm số lần một từ xuất hiện, ta chỉ quan tâm xem từ đó **có xuất hiện hay không**.

Nói ngắn gọn:
- `MultinomialNB`: dùng khi số lần xuất hiện là thông tin quan trọng.
- `BernoulliNB`: dùng khi chỉ cần biết có hay không.

Vì vậy, các giá trị khác 0 sẽ được đưa về 1 trước khi mô hình xử lý.

In [23]:
# train data
d1 = [1, 1, 1, 0, 0, 0, 0, 0, 0]
d2 = [1, 1, 0, 1, 1, 0, 0, 0, 0]
d3 = [0, 1, 0, 0, 1, 1, 0, 0, 0]
d4 = [0, 1, 0, 0, 0, 0, 1, 1, 1]

train_data = np.array([d1, d2, d3, d4])
label = np.array(['B', 'B', 'B', 'N'])

# test data
d5 = np.array([[1, 0, 0, 1, 0, 0, 0, 1, 0]])
d6 = np.array([[0, 1, 0, 0, 0, 0, 0, 1, 1]])

clf = BernoulliNB()
clf.fit(train_data, label)

print('Predicting class of d5:', str(clf.predict(d5)[0]))
print('Probability of d6 in each class:', clf.predict_proba(d6))

Predicting class of d5: B
Probability of d6 in each class: [[0.16948581 0.83051419]]


## Phần chuẩn bị cho Spam Filtering

Trong bài gốc, dữ liệu spam được lưu theo định dạng rất gọn:

Mỗi dòng của file feature có 3 số:
- số thứ nhất: chỉ số email,
- số thứ hai: chỉ số từ trong từ điển,
- số thứ ba: số lần từ đó xuất hiện trong email.

Ví dụ dòng `1 564 2` nghĩa là: email số 1 có từ thứ 564 xuất hiện 2 lần.

Cell dưới đây sẽ **tự tạo dữ liệu synthetic** theo đúng format đó nếu folder `ex6DataPrepared/` chưa có đủ file. Như vậy notebook này có thể tự chạy mà không cần file `.py` phụ.

In [24]:
data_dir = Path('ex6DataPrepared')
required_files = [
    'train-features.txt',
    'train-labels.txt',
    'test-features.txt',
    'test-labels.txt',
    'train-features-100.txt',
    'train-labels-100.txt',
    'train-features-50.txt',
    'train-labels-50.txt'
]

def build_vocab_groups(nwords=2500):
    common_words = list(range(1, 201))
    spam_words = list(range(201, 401))
    ham_words = list(range(401, 601))
    filler_words = list(range(601, nwords + 1))
    return {
        'common': common_words,
        'spam': spam_words,
        'ham': ham_words,
        'filler': filler_words
    }

def sample_email(label, groups, rng):
    if label == 1:
        primary = groups['spam']
        secondary = groups['ham']
    else:
        primary = groups['ham']
        secondary = groups['spam']

    counts = {}

    for idx in rng.sample(primary, rng.randint(8, 15)):
        counts[idx] = counts.get(idx, 0) + rng.randint(1, 4)

    for idx in rng.sample(groups['common'], rng.randint(5, 10)):
        counts[idx] = counts.get(idx, 0) + rng.randint(1, 3)

    for idx in rng.sample(secondary, rng.randint(0, 3)):
        counts[idx] = counts.get(idx, 0) + 1

    for idx in rng.sample(groups['filler'], rng.randint(3, 8)):
        counts[idx] = counts.get(idx, 0) + 1

    return dict(sorted(counts.items()))

def build_dataset(size, groups, rng):
    labels = [0] * (size // 2) + [1] * (size - size // 2)
    rng.shuffle(labels)
    emails = [sample_email(label, groups, rng) for label in labels]
    return labels, emails

def subset_balanced(labels, emails, size):
    per_class = size // 2
    selected_labels = []
    selected_emails = []
    class_counts = {0: 0, 1: 0}

    for label, email in zip(labels, emails):
        if class_counts[label] < per_class:
            selected_labels.append(label)
            selected_emails.append(email)
            class_counts[label] += 1
        if len(selected_labels) == size:
            break

    return selected_labels, selected_emails

def write_labels(path, labels):
    path.write_text('\n'.join(str(label) for label in labels) + '\n', encoding='utf-8')

def write_features(path, emails):
    lines = []
    for email_idx, features in enumerate(emails, start=1):
        for word_idx, count in features.items():
            lines.append(f'{email_idx} {word_idx} {count}')
    path.write_text('\n'.join(lines) + '\n', encoding='utf-8')

def ensure_spam_dataset():
    if all((data_dir / name).exists() for name in required_files):
        print('Dataset already exists.')
        return

    rng = random.Random(42)
    groups = build_vocab_groups(2500)
    data_dir.mkdir(parents=True, exist_ok=True)

    train_labels, train_emails = build_dataset(700, groups, rng)
    test_labels, test_emails = build_dataset(260, groups, rng)

    write_labels(data_dir / 'train-labels.txt', train_labels)
    write_features(data_dir / 'train-features.txt', train_emails)
    write_labels(data_dir / 'test-labels.txt', test_labels)
    write_features(data_dir / 'test-features.txt', test_emails)

    for subset_size in [400, 100, 50]:
        labels_subset, emails_subset = subset_balanced(train_labels, train_emails, subset_size)
        write_labels(data_dir / f'train-labels-{subset_size}.txt', labels_subset)
        write_features(data_dir / f'train-features-{subset_size}.txt', emails_subset)

    print('Synthetic dataset created in ex6DataPrepared/.')

ensure_spam_dataset()

Dataset already exists.


## Khai báo đường dẫn dữ liệu

Cell này đơn giản nhưng quan trọng: nó đặt tên các file train và test để các cell phía dưới dùng lại.

Mình giữ tên file giống bài gốc để bạn dễ đối chiếu.

In [25]:
path = 'ex6DataPrepared/'
train_data_fn = 'train-features.txt'
test_data_fn = 'test-features.txt'
train_label_fn = 'train-labels.txt'
test_label_fn = 'test-labels.txt'

## Hàm đọc dữ liệu spam

Cell này có nhiệm vụ đọc dữ liệu từ file text rồi biến nó thành ma trận đặc trưng.

Ý tưởng của hàm `read_data` là:
1. Đọc file nhãn để biết mỗi email là spam hay non-spam.
2. Đọc từng dòng của file feature.
3. Mỗi dòng có dạng `(email_id, word_id, count)`.
4. Ghi giá trị đó vào ma trận `data` có kích thước `số_email x 2500`.

Sau khi đọc xong:
- mỗi hàng là một email,
- mỗi cột là một từ trong từ điển,
- giá trị trong ô là số lần từ đó xuất hiện.

In [26]:
nwords = 2500

def read_data(data_fn, label_fn):
    with open(path + label_fn) as f:
        label = [int(x.strip()) for x in f.readlines()]

    with open(path + data_fn) as f:
        content = [x.strip() for x in f.readlines()]

    data = np.zeros((len(label), nwords), dtype=int)

    for line in content:
        email_idx, word_idx, count = [int(v) for v in line.split(' ')]
        data[email_idx - 1, word_idx - 1] = count

    return data, label

## Huấn luyện với full training set

Đây là lần chạy đầu tiên với tập train đầy đủ.

Quy trình là:
1. đọc dữ liệu train và test,
2. huấn luyện `MultinomialNB`,
3. dự đoán trên tập test,
4. tính accuracy.

Nếu dữ liệu synthetic được sinh hợp lý thì kết quả sẽ khá cao, vì từ spam và non-spam đã được tạo ra theo hai xu hướng khác nhau.

In [27]:
(train_data, train_label) = read_data(train_data_fn, train_label_fn)
(test_data, test_label) = read_data(test_data_fn, test_label_fn)

clf = MultinomialNB()
clf.fit(train_data, train_label)

y_pred = clf.predict(test_data)
print('Training size = %d, accuracy = %.2f%%' % (train_data.shape[0], accuracy_score(test_label, y_pred)*100))

Training size = 700, accuracy = 100.00%


## Thử với 100 email train

Bây giờ mình giảm lượng dữ liệu huấn luyện xuống còn `100` email để xem mô hình còn học ổn không.

Điểm hay của Naive Bayes là mô hình khá đơn giản, nên nhiều lúc dữ liệu chưa lớn lắm mà vẫn cho kết quả ổn.

In [28]:
train_data_fn = 'train-features-100.txt'
train_label_fn = 'train-labels-100.txt'
test_data_fn = 'test-features.txt'
test_label_fn = 'test-labels.txt'

(train_data, train_label) = read_data(train_data_fn, train_label_fn)
(test_data, test_label) = read_data(test_data_fn, test_label_fn)

clf = MultinomialNB()
clf.fit(train_data, train_label)
y_pred = clf.predict(test_data)
print('Training size = %d, accuracy = %.2f%%' % (train_data.shape[0], accuracy_score(test_label, y_pred)*100))

Training size = 100, accuracy = 100.00%


## Thử với 50 email train

Cell này tiếp tục giảm tập train xuống còn `50` email.

Lúc này bạn có thể so sánh trực tiếp với 2 cell trước để thấy:
- dữ liệu giảm đi bao nhiêu,
- accuracy giảm nhiều hay ít,
- và mô hình có còn giữ được tín hiệu phân lớp hay không.

In [29]:
train_data_fn = 'train-features-50.txt'
train_label_fn = 'train-labels-50.txt'
test_data_fn = 'test-features.txt'
test_label_fn = 'test-labels.txt'

(train_data, train_label) = read_data(train_data_fn, train_label_fn)
(test_data, test_label) = read_data(test_data_fn, test_label_fn)

clf = MultinomialNB()
clf.fit(train_data, train_label)
y_pred = clf.predict(test_data)
print('Training size = %d, accuracy = %.2f%%' % (train_data.shape[0], accuracy_score(test_label, y_pred)*100))

Training size = 50, accuracy = 98.85%


## So sánh thêm với BernoulliNB

Cell cuối cùng dùng cùng bộ dữ liệu vừa đọc ở trên nhưng đổi mô hình sang `BernoulliNB`.

Việc so sánh này giúp mình hiểu rõ hơn rằng:
- nếu dữ liệu thiên về **số lần xuất hiện**, `MultinomialNB` thường mạnh hơn,
- nếu dữ liệu chỉ cần biết **có hay không**, `BernoulliNB` có thể phù hợp hơn.

Với bộ dữ liệu synthetic này, `MultinomialNB` thường có lợi thế hơn vì dữ liệu được sinh theo kiểu đếm tần suất.

In [30]:
clf = BernoulliNB(binarize=.5)
clf.fit(train_data, train_label)
y_pred = clf.predict(test_data)
print('Training size = %d, accuracy = %.2f%%' % (train_data.shape[0], accuracy_score(test_label, y_pred)*100))

Training size = 50, accuracy = 96.54%
